# 02 - Baseline experiments: logistic & LightGBM

Run compact baseline experiments and produce comparison tables and plots. This notebook assumes processed artifacts are available in `data/processed/`.

In [35]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
%matplotlib inline
sns.set_theme(style='whitegrid')

In [36]:
proc = Path('../data/processed')
X = pd.read_csv(proc / 'X.csv')
y = pd.read_csv(proc / 'y.csv')
meta = pd.read_csv(proc / 'meta.csv') if (proc / 'meta.csv').exists() else pd.DataFrame()
train_idx = pd.read_csv(proc / 'train_idx.csv', header=None).iloc[:,0].tolist()
val_idx = pd.read_csv(proc / 'val_idx.csv', header=None).iloc[:,0].tolist()
test_idx = pd.read_csv(proc / 'test_idx.csv', header=None).iloc[:,0].tolist() if (proc / 'test_idx.csv').exists() else []
print('Loaded:', X.shape, y.shape)

Loaded: (43028, 10) (43028, 1)


## Quick train/val split check
Ensure indices match and show class balance in splits.

In [37]:
ycol = y.columns[0] if y.shape[1]==1 else 'is_like'
print('train size, val size, test size:', len(train_idx), len(val_idx), len(test_idx))
print('train label dist:\n', y.loc[train_idx][ycol].value_counts(normalize=True))
print('val label dist:\n', y.loc[val_idx][ycol].value_counts(normalize=True))

train size, val size, test size: 30120 6455 6456
train label dist:
 is_like
0    0.994124
1    0.005876
Name: proportion, dtype: float64
val label dist:
 is_like
0    0.995198
1    0.004802
Name: proportion, dtype: float64


## Prepare features for training
Select training columns (preprocessed).

In [38]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib
# prefer the processed training_columns file when present
proc_feat = proc / 'features_summary.json'
if proc_feat.exists():
    import json as _json
    ts = _json.loads(proc_feat.read_text())
    train_cols = ts.get('training_columns', list(X.columns))
else:
    train_cols = list(X.columns)
numeric_cols = X[train_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in train_cols if c not in numeric_cols]
numeric_cols[:10], categorical_cols[:10]

(['date',
  'hourmin',
  'is_rand',
  'tab',
  'c0',
  'c1',
  'cluster',
  'c1_videos',
  'c2',
  'cluster_videos'],
 [])

## Logistic Regression baseline
Train a compact sklearn pipeline with class-weighting.

In [39]:
from sklearn.linear_model import LogisticRegression
num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='__MISSING__')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
pre = ColumnTransformer([('num', num_pipe, numeric_cols), ('cat', cat_pipe, categorical_cols)], remainder='drop')
clf = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))])
X_train = X.loc[train_idx]
y_train = y.loc[train_idx].iloc[:,0] if y.shape[1]==1 else y.loc[train_idx]['is_like']
X_val = X.loc[val_idx]
y_val = y.loc[val_idx].iloc[:,0] if y.shape[1]==1 else y.loc[val_idx]['is_like']
clf.fit(X_train, y_train)
val_prob = clf.predict_proba(X_val)[:,1]
joblib.dump(clf, '../artifacts/models/logistic_pipeline.joblib')
print('Done')

Done


## LightGBM baseline
Train LightGBM via sklearn API inside a pipeline.

In [40]:
from lightgbm import LGBMClassifier
clf_lgb = Pipeline([('pre', pre), ('clf', LGBMClassifier(n_estimators=100))])
clf_lgb.fit(X_train, y_train)
val_prob_lgb = clf_lgb.predict_proba(X_val)[:,1]
joblib.dump(clf_lgb, '../artifacts/models/lightgbm_pipeline.joblib')
print('LightGBM done')

[LightGBM] [Info] Number of positive: 177, number of negative: 29943
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000475 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1089
[LightGBM] [Info] Number of data points in the train set: 30120, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.005876 -> initscore=-5.130901
[LightGBM] [Info] Start training from score -5.130901
LightGBM done


c:\Users\hamza\DLP\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Comparison table
Compute metrics for both models and show a compact table.

In [41]:
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, accuracy_score, precision_score, recall_score, f1_score
def metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        'roc_auc': roc_auc_score(y_true, y_prob),
        'pr_auc': average_precision_score(y_true, y_prob),
        'log_loss': log_loss(y_true, np.clip(y_prob,1e-15,1-1e-15)),
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
    }
m_log = metrics(y_val, val_prob)
m_lgb = metrics(y_val, val_prob_lgb)
pd.DataFrame({'logistic': m_log, 'lightgbm': m_lgb})

,logistic,lightgbm
roc_auc,0.552128,0.744903
pr_auc,0.006364,0.016204
log_loss,0.633905,0.033310
accuracy,0.647560,0.995043
precision,0.005727,0.000000
recall,0.419355,0.000000
f1,0.011299,0.000000


## Feature importance (LightGBM)
Plot feature importances from the LightGBM model (if available).

In [ ]:
# attempt to extract feature names after preprocessing
preproc = clf_lgb.named_steps['pre']
# numeric feature names: find the 'num' transformer columns if present
num_names = []
for t in getattr(preproc, 'transformers_', []):
    if t[0] == 'num':
        num_names = list(t[2])
        break
# categorical OHE: safely access named_transformers_ and inner pipeline
cat_ohe = None
if hasattr(preproc, 'named_transformers_') and 'cat' in preproc.named_transformers_:
    cat_transformer = preproc.named_transformers_['cat']
    if hasattr(cat_transformer, 'named_steps') and 'ohe' in cat_transformer.named_steps:
        cat_ohe = cat_transformer.named_steps['ohe']
cat_names = []
if cat_ohe is not None:
    # find the categorical column list from the transformers_ tuples
    cat_cols = []
    for t in getattr(preproc, 'transformers_', []):
        if t[0] == 'cat':
            cat_cols = list(t[2])
            break
    try:
        cat_names = cat_ohe.get_feature_names_out(cat_cols).tolist()
    except Exception:
        cat_names = []
feature_names = list(num_names) + cat_names
try:
    lgbm = clf_lgb.named_steps['clf']
    importances = lgbm.feature_importances_
    imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False).head(30)
    imp_df.plot.barh(x='feature', y='importance', figsize=(8,10))
except Exception as e:
    print('Could not extract importances:', e)

ValueError: dictionary update sequence element #0 has length 3; 2 is required

## Precision-Recall curve & ROC curve
Plot PR and ROC for the two baselines.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve, auc
plt.figure(figsize=(6,5))
p, r, _ = precision_recall_curve(y_val, val_prob)
plt.plot(r, p, label='logistic')
p2, r2, _ = precision_recall_curve(y_val, val_prob_lgb)
plt.plot(r2, p2, label='lightgbm')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend()
plt.title('Precision-Recall')
plt.show()
plt.figure(figsize=(6,5))
fpr, tpr, _ = roc_curve(y_val, val_prob)
plt.plot(fpr, tpr, label=f'logistic (AUC={auc(fpr,tpr):.3f})')
fpr2, tpr2, _ = roc_curve(y_val, val_prob_lgb)
plt.plot(fpr2, tpr2, label=f'lightgbm (AUC={auc(fpr2,tpr2):.3f})')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.legend()
plt.title('ROC Curve')
plt.show()

## Top-k ranking summary
Show precision@k aggregated by user for a few k values.

In [ ]:
def precision_at_k_by_user(meta, y_true, y_prob, k=5, user_col='user_id'):
    if user_col not in meta.columns:
        order = np.argsort(-y_prob)
        topk = order[:k]
        return float(np.mean(np.array(y_true)[topk]))
    df = pd.DataFrame({'user_id': meta[user_col].values, 'y_true': y_true.values, 'y_prob': y_prob})
    precisions = []
    for uid, g in df.groupby('user_id'):
        g_sorted = g.sort_values('y_prob', ascending=False)
        top = g_sorted.head(k)
        if len(top)==0: continue
        precisions.append(top['y_true'].mean())
    return float(np.nanmean(precisions)) if precisions else 0.0
for k in [1,5,10]:
    print('logistic prec@%d:'%k, precision_at_k_by_user(meta.loc[val_idx] if not meta.empty else pd.DataFrame(), y_val, val_prob, k=k))
    print('lightgbm prec@%d:'%k, precision_at_k_by_user(meta.loc[val_idx] if not meta.empty else pd.DataFrame(), y_val, val_prob_lgb, k=k))